# NB6: Label Interpolation Help/Hurt Case Examples

Extract concrete examples where retrieval (label interpolation + aux loss) differs from no-retrieval:
- **Help cases:** retrieval correct, no-retrieval wrong
- **Hurt cases:** no-retrieval correct, retrieval wrong

Each case includes the retrieved neighbors (sentence, polarity, cosine score) to show WHY label interpolation helped or hurt.

**Input:** `p5-nb1-stage1`, `p5-embed-v4`, `stage2-sem14-eval-checkpoints`

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
import torch, gc, math
import numpy as np
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Wire Checkpoints & Data

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}', f'/kaggle/input/datasets/lcminhc/{name}',
              f'/kaggle/input/datasets/duclm318/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
EMB = find_input('p5-embed-v4')
EVAL_CKPT = find_input('stage2-sem14-eval-checkpoints')

print(f'NB1: {NB1} -> {os.listdir(NB1)}')
print(f'EMB: {EMB} -> {os.listdir(EMB)}')
print(f'EVAL_CKPT: {EVAL_CKPT} -> {os.listdir(EVAL_CKPT)}')

# Stage 1
os.makedirs('checkpoints/stage1', exist_ok=True)
shutil.copy(f'{NB1}/stage1_2014_cataware_best.pt', 'checkpoints/stage1/best.pt')

# Stage 2 — Retrieval + Aux Loss
os.makedirs('checkpoints/stage2_2014', exist_ok=True)
shutil.copy(f'{EVAL_CKPT}/stage2_2014_auxloss.pt', 'checkpoints/stage2_2014/best.pt')

# Stage 2 — No-Retrieval
os.makedirs('checkpoints/stage2_2014_noret', exist_ok=True)
shutil.copy(f'{EVAL_CKPT}/stage2_2014_noret.pt', 'checkpoints/stage2_2014_noret/best.pt')

# Embedding
os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')

# Data
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed/category_detection.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records.jsonl')

# Build FAISS index
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

print('\nAll files wired.')

## 2. Load Data & Models

In [ ]:
from src.utils.io import load_yaml, read_jsonl
from src.utils.seed import set_seed
from src.data.category_builder import CATEGORY_LIST, CAT2IDX, POL2ID
from src.embedding.model import ContrastiveEmbedder
from src.retrieval.index import load_index
from src.retrieval.retriever import Retriever
from transformers import AutoTokenizer

ID2POL = {v: k for k, v in POL2ID.items()}
set_seed(42)

s1_cfg = load_yaml('configs/stage1_2014_cataware.yaml')
s2_cfg = load_yaml('configs/stage2_2014_auxloss.yaml')
s2_noret_cfg = load_yaml('configs/stage2_2014_noret.yaml')
ret_cfg = load_yaml('configs/retrieval_v2.yaml')

cat_records = read_jsonl(s1_cfg['category_path'])
sent_records = read_jsonl(s2_cfg['sentiment_path'])

test_cat = [r for r in cat_records if r['split'] == 'test']
test_sent = [r for r in sent_records if r['split'] == 'test']
train_sent = [r for r in sent_records if r['split'] == 'train']

print(f'Test cat records: {len(test_cat)}')
print(f'Test sent records: {len(test_sent)}')
print(f'Train sent records: {len(train_sent)}')

In [ ]:
# Load embedding model + FAISS index + retriever
embed_model = ContrastiveEmbedder(model_name=s2_cfg['model_name'], proj_dim=256)
embed_model.load_state_dict(
    torch.load('checkpoints/embedding_2014/best.pt', map_location='cpu'), strict=False)
embed_model.to(DEVICE)
embed_model.eval()

full_index, full_metadata, full_vectors = load_index('indexes')
retriever = Retriever(full_index, full_metadata,
                      top_k=ret_cfg['top_k'], threshold=ret_cfg['threshold'])
tokenizer = AutoTokenizer.from_pretrained(s2_cfg['model_name'])

print(f'Embedding model loaded')
print(f'FAISS index: {full_index.ntotal} vectors')
print(f'Retriever: top_k={ret_cfg["top_k"]}, threshold={ret_cfg["threshold"]}')

## 3. Stage 1 — Predict Categories

In [ ]:
from src.absa.category_model import CategoryDetector
from src.absa.category_dataset import CategoryDataset
from src.absa.category_trainer import _tune_global_threshold, _apply_global_threshold
from torch.utils.data import DataLoader

def collect_logits(model, dataset, device, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size)
    all_logits = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            out = model(batch['input_ids'], batch['attention_mask'])
            all_logits.append(out['logits'].cpu())
    return torch.cat(all_logits, dim=0)

# Load Stage 1
s1_model = CategoryDetector(
    model_name=s1_cfg['model_name'],
    num_categories=s1_cfg['num_categories'],
    use_cat_attention=s1_cfg.get('use_cat_attention', False),
).to(DEVICE)
s1_ckpt = torch.load('checkpoints/stage1/best.pt', map_location=DEVICE)
s1_model.load_state_dict(s1_ckpt['model_state'], strict=False)
s1_model.eval()

# Tune threshold on val
train_cat = [r for r in cat_records if r['split'] == 'train']
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    label_matrix = np.array([r['category_vector'] for r in train_cat])
    msss = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=s1_cfg['val_ratio'],
        random_state=s1_cfg['seed'])
    for _, val_idx in msss.split(label_matrix, label_matrix):
        val_cat = [train_cat[i] for i in val_idx]
except ImportError:
    stratify_key = [min(sum(r['category_vector']), 2) for r in train_cat]
    _, val_cat = train_test_split(
        train_cat, test_size=s1_cfg['val_ratio'],
        random_state=s1_cfg['seed'], stratify=stratify_key)

val_ds = CategoryDataset(val_cat, tokenizer_name=s1_cfg['model_name'],
                         max_length=s1_cfg['max_seq_length'])
val_logits = collect_logits(s1_model, val_ds, DEVICE)
val_labels = torch.stack([torch.tensor(r['category_vector'], dtype=torch.float32)
                          for r in val_cat])
threshold = _tune_global_threshold(val_logits, val_labels)
print(f'Tuned global threshold: {threshold:.2f}')

# Predict on test
test_ds = CategoryDataset(test_cat, tokenizer_name=s1_cfg['model_name'],
                          max_length=s1_cfg['max_seq_length'])
test_logits = collect_logits(s1_model, test_ds, DEVICE)
pred_cats_list = _apply_global_threshold(test_logits, threshold)

from src.evaluation.category_metrics import category_f1
gold_cats_list = [set(cr['categories']) for cr in test_cat]
cat_m = category_f1(pred_cats_list, gold_cats_list)
print(f'Stage 1 Cat F1: {cat_m["f1"]:.4f}')

del s1_model, s1_ckpt
gc.collect()
torch.cuda.empty_cache()
print('Stage 1 model freed.')

## 4. Run Both Stage 2 Models

In [ ]:
from src.absa.sentiment_dataset import SentimentDataset
from src.absa.sentiment_model import SentimentPredictor

def predict_sentiment_with_details(model, records, retriever, embedding_model,
                                   tokenizer_name, max_length, top_k, device,
                                   use_retrieval, store_vectors=None, batch_size=16):
    """Run inference, return predictions and softmax probabilities."""
    embed_device = 'cpu'
    if embedding_model is not None:
        embed_device = next(embedding_model.parameters()).device.type
    ds = SentimentDataset(
        records, retriever=retriever,
        tokenizer_name=tokenizer_name,
        embedding_model=embedding_model,
        store_vectors=store_vectors,
        max_length=max_length,
        top_k=top_k, device=embed_device,
        use_retrieval=use_retrieval,
    )
    loader = DataLoader(ds, batch_size=batch_size)
    all_preds = []
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in batch.items()}
            out = model(
                input_ids=batch_gpu['input_ids'],
                attention_mask=batch_gpu['attention_mask'],
                neighbor_polarities=batch_gpu.get('neighbor_polarities'),
                neighbor_scores=batch_gpu.get('neighbor_scores'),
                query_vec=batch_gpu.get('query_vec'),
                neighbor_vecs=batch_gpu.get('neighbor_vecs'),
                query_polarity=batch_gpu.get('query_polarity'),
            )
            logits = out['logits']
            probs = torch.softmax(logits, dim=-1).cpu()
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_probs.append(probs)
    all_probs = torch.cat(all_probs, dim=0)
    return all_preds, all_probs

In [ ]:
# Build Stage 2 records from predicted categories
stage2_pred_records = []
for cr, pred_cats in zip(test_cat, pred_cats_list):
    for cat in sorted(pred_cats):
        stage2_pred_records.append({
            'id': f"{cr['sentence_id']}_{cat}",
            'sentence': cr['sentence'],
            'category': cat,
            'polarity': 'positive',
            'split': 'test',
        })
print(f'Stage 2 records from predicted categories: {len(stage2_pred_records)}')

In [ ]:
# --- Retrieval model ---
s2_ret_model = SentimentPredictor(
    model_name=s2_cfg['model_name'],
    num_sent_labels=s2_cfg['num_sent_labels'],
    embed_dim=s2_cfg.get('embed_dim', 64),
    tau=s2_cfg.get('tau', 0.05),
    dropout=s2_cfg.get('dropout', 0.1),
    use_retrieval=True,
    use_learnable_retriever=s2_cfg.get('use_learnable_retriever', False),
    margin=s2_cfg.get('rank_margin', 0.1),
    w_mode=s2_cfg.get('w_mode', 'full'),
    w_rank=s2_cfg.get('w_rank', 16),
).to(DEVICE)
s2_ret_state = torch.load('checkpoints/stage2_2014/best.pt', map_location=DEVICE)
s2_ret_model.load_state_dict(s2_ret_state, strict=False)
s2_ret_model.eval()
print('Retrieval model loaded')

print('Running retrieval inference...')
ret_preds, ret_probs = predict_sentiment_with_details(
    s2_ret_model, stage2_pred_records, retriever, embed_model,
    tokenizer_name=s2_cfg['model_name'],
    max_length=s2_cfg['max_seq_length'],
    top_k=ret_cfg['top_k'],
    device=DEVICE, use_retrieval=True,
    store_vectors=full_vectors)
print(f'  {len(ret_preds)} predictions')

del s2_ret_model, s2_ret_state
gc.collect()
torch.cuda.empty_cache()
print('Retrieval model freed.')

In [ ]:
# --- No-retrieval model ---
s2_noret_model = SentimentPredictor(
    model_name=s2_noret_cfg['model_name'],
    num_sent_labels=s2_noret_cfg['num_sent_labels'],
    embed_dim=s2_noret_cfg.get('embed_dim', 64),
    tau=s2_noret_cfg.get('tau', 0.05),
    dropout=s2_noret_cfg.get('dropout', 0.1),
    use_retrieval=False,
).to(DEVICE)
s2_noret_state = torch.load('checkpoints/stage2_2014_noret/best.pt', map_location=DEVICE)
s2_noret_model.load_state_dict(s2_noret_state, strict=False)
s2_noret_model.eval()
print('No-retrieval model loaded')

print('Running no-retrieval inference...')
noret_preds, noret_probs = predict_sentiment_with_details(
    s2_noret_model, stage2_pred_records, None, None,
    tokenizer_name=s2_noret_cfg['model_name'],
    max_length=s2_noret_cfg['max_seq_length'],
    top_k=0,
    device=DEVICE, use_retrieval=False)
print(f'  {len(noret_preds)} predictions')

del s2_noret_model, s2_noret_state
gc.collect()
torch.cuda.empty_cache()
print('No-retrieval model freed.')

## 5. Find Help & Hurt Cases

In [ ]:
# Build gold lookup
gold_by_cat_per_sent = {}
for r in test_sent:
    key = r['sentence']
    if key not in gold_by_cat_per_sent:
        gold_by_cat_per_sent[key] = {}
    gold_by_cat_per_sent[key][r['category']] = r['polarity']

help_cases = []
hurt_cases = []

for i, rec in enumerate(stage2_pred_records):
    sent = rec['sentence']
    cat = rec['category']

    gold_pol = gold_by_cat_per_sent.get(sent, {}).get(cat)
    if gold_pol is None:
        continue

    ret_pol = ID2POL[ret_preds[i]]
    noret_pol = ID2POL[noret_preds[i]]

    if ret_pol == gold_pol and noret_pol != gold_pol:
        help_cases.append({
            'idx': i, 'sentence': sent, 'category': cat,
            'gold': gold_pol, 'ret_pred': ret_pol, 'noret_pred': noret_pol,
            'ret_conf': ret_probs[i].max().item(),
            'noret_conf': noret_probs[i].max().item(),
            'ret_probs': {ID2POL[j]: round(ret_probs[i][j].item(), 4) for j in range(3)},
            'noret_probs': {ID2POL[j]: round(noret_probs[i][j].item(), 4) for j in range(3)},
        })
    elif noret_pol == gold_pol and ret_pol != gold_pol:
        hurt_cases.append({
            'idx': i, 'sentence': sent, 'category': cat,
            'gold': gold_pol, 'ret_pred': ret_pol, 'noret_pred': noret_pol,
            'ret_conf': ret_probs[i].max().item(),
            'noret_conf': noret_probs[i].max().item(),
            'ret_probs': {ID2POL[j]: round(ret_probs[i][j].item(), 4) for j in range(3)},
            'noret_probs': {ID2POL[j]: round(noret_probs[i][j].item(), 4) for j in range(3)},
        })

print(f'HELP cases (ret correct, no-ret wrong): {len(help_cases)}')
print(f'HURT cases (no-ret correct, ret wrong): {len(hurt_cases)}')
print(f'Net effect of retrieval: {len(help_cases) - len(hurt_cases):+d} samples')

In [ ]:
# Summary by gold polarity and error direction
print('\n=== HELP cases by gold polarity ===')
for pol in ['positive', 'negative', 'neutral']:
    subset = [c for c in help_cases if c['gold'] == pol]
    if not subset:
        print(f'  {pol}: 0')
        continue
    directions = Counter(c['noret_pred'] for c in subset)
    dir_str = ', '.join(f'noret->{p}: {n}' for p, n in directions.most_common())
    print(f'  {pol}: {len(subset)}  ({dir_str})')

print('\n=== HURT cases by gold polarity ===')
for pol in ['positive', 'negative', 'neutral']:
    subset = [c for c in hurt_cases if c['gold'] == pol]
    if not subset:
        print(f'  {pol}: 0')
        continue
    directions = Counter(c['ret_pred'] for c in subset)
    dir_str = ', '.join(f'ret->{p}: {n}' for p, n in directions.most_common())
    print(f'  {pol}: {len(subset)}  ({dir_str})')

## 6. Retrieve Neighbors for All Cases

In [ ]:
def attach_neighbors(cases, retriever, embed_model, tokenizer, device):
    """For each case, retrieve neighbors and attach details."""
    for case in cases:
        tok_enc = tokenizer(
            case['sentence'], case['category'],
            max_length=128, padding=False, truncation=True,
            return_tensors='pt')
        with torch.no_grad():
            query_vec = embed_model.encode(
                tok_enc['input_ids'].to(device),
                tok_enc['attention_mask'].to(device))
        query_np = query_vec.cpu().numpy().astype('float32')
        neighbors = retriever.retrieve(query_np, exclude_sentence=case['sentence'])

        case['neighbors'] = [{
            'sentence': nb['sentence'],
            'category': nb.get('aspect_category', nb.get('category')),
            'polarity': nb['polarity'],
            'score': round(nb['score'], 4),
        } for nb in neighbors]

        if neighbors:
            case['pol_match'] = sum(
                1 for nb in neighbors if nb['polarity'] == case['gold']
            ) / len(neighbors)
        else:
            case['pol_match'] = 0.0


print(f'Retrieving neighbors for {len(help_cases)} help cases...')
attach_neighbors(help_cases, retriever, embed_model, tokenizer, DEVICE)

print(f'Retrieving neighbors for {len(hurt_cases)} hurt cases...')
attach_neighbors(hurt_cases, retriever, embed_model, tokenizer, DEVICE)

print('Done.')

## 7. Display All Cases

In [ ]:
def print_cases(cases, case_type):
    """Print all cases grouped by gold polarity."""
    label = 'HELP' if case_type == 'help' else 'HURT'
    wrong_model = 'No-Ret' if case_type == 'help' else 'Retrieval'
    right_model = 'Retrieval' if case_type == 'help' else 'No-Ret'

    print(f'\n{"+" * 80}')
    print(f'  {label} CASES  ({right_model} correct, {wrong_model} wrong)  —  {len(cases)} total')
    print(f'{"+" * 80}')

    for pol in ['positive', 'negative', 'neutral']:
        subset = [c for c in cases if c['gold'] == pol]
        if not subset:
            continue

        print(f'\n{"=" * 80}')
        print(f'  Gold = {pol.upper()}  ({len(subset)} cases)')
        print(f'{"=" * 80}')

        for j, case in enumerate(subset):
            print(f'\n--- {label} #{j+1} ---')
            print(f'  Sentence:  {case["sentence"]}')
            print(f'  Category:  {case["category"]}')
            print(f'  Gold:      {case["gold"]}')

            if case_type == 'help':
                print(f'  Ret pred:  {case["ret_pred"]}  (CORRECT)  conf={case["ret_conf"]:.4f}  probs={case["ret_probs"]}')
                print(f'  NoRet pred:{case["noret_pred"]}  (WRONG)    conf={case["noret_conf"]:.4f}  probs={case["noret_probs"]}')
            else:
                print(f'  Ret pred:  {case["ret_pred"]}  (WRONG)    conf={case["ret_conf"]:.4f}  probs={case["ret_probs"]}')
                print(f'  NoRet pred:{case["noret_pred"]}  (CORRECT)  conf={case["noret_conf"]:.4f}  probs={case["noret_probs"]}')

            print(f'  pol_match: {case["pol_match"]:.2f} ({sum(1 for nb in case["neighbors"] if nb["polarity"] == case["gold"])}/{len(case["neighbors"])} neighbors match gold)')
            for k, nb in enumerate(case.get('neighbors', [])):
                match_marker = 'v' if nb['polarity'] == case['gold'] else 'x'
                print(f'    nb{k+1} [{match_marker}] [{nb["polarity"]:8s}] (sim={nb["score"]:.4f}) {nb["sentence"]}')


print_cases(help_cases, 'help')

In [ ]:
print_cases(hurt_cases, 'hurt')

In [ ]:
# Final summary
print('\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print(f'HELP cases (retrieval correct, no-ret wrong): {len(help_cases)}')
print(f'HURT cases (no-ret correct, retrieval wrong): {len(hurt_cases)}')
print(f'Net effect: {len(help_cases) - len(hurt_cases):+d}')

print(f'\nHELP — mean pol_match: {np.mean([c["pol_match"] for c in help_cases]):.4f}' if help_cases else '')
print(f'HURT — mean pol_match: {np.mean([c["pol_match"] for c in hurt_cases]):.4f}' if hurt_cases else '')

print(f'\nHELP — mean ret confidence:   {np.mean([c["ret_conf"] for c in help_cases]):.4f}' if help_cases else '')
print(f'HELP — mean noret confidence: {np.mean([c["noret_conf"] for c in help_cases]):.4f}' if help_cases else '')
print(f'HURT — mean ret confidence:   {np.mean([c["ret_conf"] for c in hurt_cases]):.4f}' if hurt_cases else '')
print(f'HURT — mean noret confidence: {np.mean([c["noret_conf"] for c in hurt_cases]):.4f}' if hurt_cases else '')

print(f'\nBy gold polarity:')
print(f'{"Polarity":<12} {"HELP":>6} {"HURT":>6} {"Net":>6}')
print(f'{"-"*36}')
for pol in ['positive', 'negative', 'neutral']:
    h = len([c for c in help_cases if c['gold'] == pol])
    u = len([c for c in hurt_cases if c['gold'] == pol])
    print(f'{pol:<12} {h:>6} {u:>6} {h-u:>+6}')